In [ ]:
# ==============================================================
# Proyecto Final - Ciencia Computacional Intermedia (Tercer corte)
# Confiabilidad y planeación de capacidad de un servicio de inferencia de IA
# Grupo 1: Henry Cantor, Santiago Fajardo, Sebastián Páez, Sebastián Acuña
# ==============================================================
# Identificación del grupo (requisito del entregable E1).
# El grupo tiene 4 integrantes, así que cada C es la suma de las cédulas de una pareja:
#   C1 = Henry Cantor + Santiago Fajardo
#   C2 = Sebastián Páez + Sebastián Acuña
C1 = 2125981045
C2 = 2154025811
S = C1 + C2

print(f"C1 = {C1}")
print(f"C2 = {C2}")
print(f"S  = {S}")


In [ ]:
# Librerías permitidas por el enunciado (Sección 6.1)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize

# Única fuente de aleatoriedad de todo el notebook (requisito E1)
rng = np.random.default_rng(S)

# Parámetros personalizados del grupo (Sección 3 del enunciado)
tau_ms = 180 + 10 * (S % 9)                    # umbral de latencia del SLO (ms)
epsilon = 0.01 * (1 + S % 3)                   # proporción máxima tolerada con L > tau
alpha = {0: 0.10, 1: 0.05, 2: 0.01}[S % 3]     # nivel de confianza = 1 - alpha
nivel_confianza = 1 - alpha
g = round(1.4 + 0.1 * (S % 7), 2)              # factor de crecimiento de la tasa de llegadas
t1_kb = 80 + 10 * (S % 5)                      # frontera pequeña / mediana (KB)
t2_kb = 400 + 25 * (S % 5)                     # frontera mediana / grande (KB)
pi_incidentes = (4 + S % 6) / 1000             # prevalencia de incidentes en producción (por minuto)
k_max = 4 + (S % 5)                            # máximo de falsas alarmas por día toleradas
r = S % 6                                      # desplazamiento entero (ítem B4)
N = 10 + (S % 11)                              # solicitudes consecutivas (ítem C4)
c0 = 2                                         # GPUs del pool real

parametros = pd.DataFrame({
    "Símbolo": ["tau (ms)", "epsilon", "1 - alpha", "g", "t1 (KB)", "t2 (KB)", "pi", "k_max", "r", "N", "c0"],
    "Valor": [tau_ms, epsilon, nivel_confianza, g, t1_kb, t2_kb, pi_incidentes, k_max, r, N, c0],
})
parametros


In [ ]:
# Estilo y utilidades comunes para cumplir las normas de la Sección 6.1:
# título, ejes con unidades, leyenda si hay más de una serie, resolución legible
# y un pie de figura que diga qué debe concluir el lector.
import os

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

CARPETA_FIGURAS = "figuras"
os.makedirs(CARPETA_FIGURAS, exist_ok=True)


def pie_de_figura(fig, numero, conclusion):
    """Agrega debajo de la figura el pie 'Figura n. <conclusión para el lector>'."""
    fig.text(0.5, -0.02, f"Figura {numero}. {conclusion}",
             ha="center", va="top", wrap=True, fontsize=10, style="italic")


def guardar_figura(fig, numero):
    """Guarda la figura como figuras/figura_<n>.png para insertarla en el informe."""
    ruta = os.path.join(CARPETA_FIGURAS, f"figura_{numero}.png")
    fig.savefig(ruta, bbox_inches="tight")
    return ruta


def formatear_intervalo(estimacion, inferior, superior, metodo, decimales=3):
    """Texto de un intervalo con estimación puntual, extremos, nivel y método (Sección 6.1)."""
    d = decimales
    return (f"{estimacion:.{d}f}  [{inferior:.{d}f}, {superior:.{d}f}]  "
            f"(IC {nivel_confianza:.0%}, método: {metodo})")


In [ ]:
# Carga del paquete de datos único del grupo (entregado por el docente).
# Suban los CSV al panel de Archivos de Colab, o monten Google Drive y cambien RUTA_DATOS.
from pathlib import Path

RUTA_DATOS = Path(".")
ARCHIVOS = {
    "solicitudes": "solicitudes.csv",
    "monitoreo": "monitoreo.csv",
    # "validacion": "solicitudes_validacion.csv",  # se libera el día 9; solo se usa en el ítem F6
}

datos = {}
for nombre, archivo in ARCHIVOS.items():
    ruta = RUTA_DATOS / archivo
    if ruta.exists():
        datos[nombre] = pd.read_csv(ruta)
        print(f"{archivo}: {datos[nombre].shape[0]} filas x {datos[nombre].shape[1]} columnas")
    else:
        print(f"Falta {archivo} en {ruta.resolve().parent}")


# Parte A - Auditoría y preparación de los datos (8 pts)

### A1. Auditoría de `solicitudes.csv`
Inconsistencias encontradas y su tratamiento. Tabla "filas afectadas / criterio / tratamiento".

### A2. ¿Un único régimen?
Criterio estadístico cuantitativo, segmentos, sensibilidad al criterio y régimen estable que se usará en adelante.

### A3. Resumen descriptivo
Centro, dispersión, asimetría y curtosis de latencia_ms, gpu_ms, red_ms y tam_kb, por versión y por régimen.

**Figura 1:** evolución temporal del sistema.

# Parte B - Llegadas, fallos y reintentos (14 pts)

### B1. Modelo del proceso de llegadas
Tiempos entre llegadas y conteos por intervalo. Validación por al menos dos vías cuantitativas; estimaciones puntuales e intervalos de los parámetros.

### B2. Bloques de 100 solicitudes
Modelo del número de solicitudes por bloque que requirieron más de un intento; contraste con ambos regímenes.

### B3. Modelo de la variable intentos
Estimador de máxima verosimilitud (deducción manuscrita en E3), estimación con su intervalo y simulación para evaluar el estimador basado en el promedio de intentos.

### B4. Reintentos en una ventana de 10 s
Probabilidad de que al menos k = ceil(mu) + r solicitudes requieran más de un intento: cálculo analítico y verificación por Montecarlo.

# Parte C - Tiempo de cómputo y tiempo de red (14 pts)

### C1. Distribución de G (todas las solicitudes)
Función de distribución acumulada, naturaleza de la variable, E[G] y Var(G) frente a los valores muestrales (deducción manuscrita en E3).

**Figura 2.**

### C2. Máxima verosimilitud para la parte continua de G
Log-verosimilitud escrita y optimizada numéricamente por el grupo (no basta con .fit()), comparación con otro método y bondad de ajuste gráfica y numérica.

### C3. ¿red_ms admite un modelo normal?
Al menos dos criterios, P(red > x) para tres valores de x y percentil 99,9 % con intervalo.

**Figura 3.**

### C4. Tiempo total de GPU de N solicitudes no cacheadas
P(T_N > N (m + 15)) con al menos dos aproximaciones analíticas, comparadas con una simulación.

# Parte D - Dependencias y esperanza condicional (14 pts)

### D1. Conjunta (clase de tamaño, cache_hit)
Marginales, condicionales e independencia con evidencia e intervalos.

**Figura 4** (mapa de calor).

### D2. Esperanza condicional
E[G | clase, no caché] y E[G | tam_kb] no paramétricas; leyes de la esperanza total y de la varianza total; fracción de Var(G) explicada por la clase.

### D3. Modelo lineal para E[G | tam_kb, no caché]
Mínimos cuadrados resolviendo el sistema matricial por cuenta propia (sin sklearn ni statsmodels; deducción manuscrita en E3), intervalos de los coeficientes, análisis de residuos y corrección si se viola algún supuesto.

**Figura 5.**

### D4. Covarianza y correlación entre tam_kb y gpu_ms (no cacheadas)
Interpretación en relación con D2 y D3.

# Parte E - Monitoreo y decisión de alertas (12 pts)

### E1. Sensibilidad y tasa de falsos positivos
Para D1, D2, "D1 y D2" y "D1 o D2", con intervalos de confianza.

### E2. Probabilidad de incidente dada cada regla en producción
Usando la prevalencia pi del grupo; por qué difiere de lo que se vería directamente en monitoreo.csv.

### E3. ¿D1 y D2 son condicionalmente independientes?
Verificación con datos y efecto de la conclusión sobre E2.

### E4. Falsas alarmas por día
Qué reglas cumplen P(falsas alarmas en 1440 min <= k_max) >= 0,90, propagando la incertidumbre, y regla recomendada.

**Figura 6.**

# Parte F - Inferencia, distribuciones muestrales y comparación de versiones (16 pts)

### F1. Distribución muestral de la media de latencia_ms
Para n en {5, 30, 200, 2000}: forma, asimetría, curtosis, dispersión y menor n con cobertura efectiva >= (1 - alpha) - 0,01.

**Figura 7.**

### F2. Intervalo para la latencia media
Evaluación crítica de los supuestos del procedimiento clásico y, si no se cumplen, corrección y su efecto sobre el intervalo.

### F3. Versión B frente a versión A
(i) tiempo medio de GPU no cacheado, (ii) probabilidad de más de un intento, (iii) probabilidad de superar tau. Intervalos de las tres diferencias, comparabilidad de los grupos, conclusiones conjuntas y decisión.

### F4. Cobertura de intervalos para una proporción
Al menos dos procedimientos, >= 10^4 réplicas por configuración, variando n y la proporción verdadera.

**Figura 8.**

### F5. Tamaño de muestra por versión
Para que el intervalo de F3(i) tenga semiancho <= 1 ms (deducción manuscrita en E3) y verificación por simulación.

### F6. Validación con solicitudes_validacion.csv (día 9)
Qué modelos e intervalos de B, C y F siguen siendo válidos, qué parámetros cambiaron y qué implica para la Parte G.

# Parte G - Simulación y decisión de capacidad (14 pts)

### G1. Simulador de Montecarlo con c GPUs
Implementado por el grupo (sin librerías de simulación de eventos discretos) y validado con c = c0 = 2 en el régimen estable.

### G2. Capacidad con la tasa de llegadas multiplicada por g
P(L > tau) para c = 1, 2, ... con intervalos a partir de réplicas independientes, menor c que cumple el SLO y estabilidad según la carga ofrecida.

**Figura 9.**

### G3. Robustez de c*
Propagación de la incertidumbre de los parámetros por Montecarlo, distribución de c* y contraste con un modelo analítico simplificado.

### G4. Memorando ejecutivo
Se entrega aparte como PDF de una página (entregable E4). Cada cifra del memorando debe poder rastrearse a una celda de este notebook.